In [1]:
import pandas as pd
import numpy as np

In [2]:
df_mosca=pd.read_csv('Asistencia Mosca  - ASISTENCIA 2024.csv')


In [4]:
df_mosca.columns=df_mosca.columns.str.strip()
df_mosca.columns=df_mosca.columns.str.lower()
df_mosca = df_mosca.rename(columns={'quorum?': 'quorum'})
df_mosca = df_mosca.rename(columns={'comidas': 'comida'})
df_mosca = df_mosca.rename(columns={'tema de la noche': 'tema'})
df_mosca.drop(columns='unnamed: 17',inplace=True)



In [5]:
df_mosca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fecha               44 non-null     object 
 1   axel zielonka       44 non-null     object 
 2   federico saroka     44 non-null     object 
 3   federico cabelli    44 non-null     object 
 4   federico koltan     44 non-null     object 
 5   manuel hirsch       44 non-null     object 
 6   joaquín sokolowicz  44 non-null     object 
 7   lucas rotmitrovsky  44 non-null     float64
 8   gonzalo borinsky    44 non-null     object 
 9   matias nemirovsky   44 non-null     object 
 10  ianick izon         44 non-null     object 
 11  santiago tabak      44 non-null     object 
 12  quorum              44 non-null     object 
 13  comida              41 non-null     object 
 14  precio              39 non-null     object 
 15  casa                42 non-null     object 
 16  tema      

In [6]:
df_mosca.replace({'Presente': 1, 'Presente/2': 1, 'Presente ': 1}, inplace=True)

In [7]:
columnas_cena = ['fecha', 'quorum', 'comida', 'precio', 'casa', 'tema']
df_cena = df_mosca[columnas_cena].copy()
df_cena.casa.replace('-',np.nan)
df_cena.casa.unique()
df_cena["fecha"] = pd.to_datetime(df_cena["fecha"], dayfirst=True, errors="raise")

df_cena['id_cena'] = range(1, len(df_cena) + 1)
df_cena['precio'] = df_cena['precio'].str.replace('$', '', regex=False)
df_cena['precio'] = df_cena['precio'].str.replace(',', '', regex=False)
df_cena['precio'] = df_cena['precio'].replace('-', np.nan)
df_cena['comida'] = df_cena['comida'].replace('-', np.nan)
df_cena.precio.astype(float)
df_cena['casa'] = df_cena['casa'].str.capitalize().str.strip()
df_cena['comida'] = df_cena['comida'].str.capitalize().str.strip()
df_cena['casa'] = df_cena['casa'].replace({
    'Axel': 'Axel cdc',
    'Gonzi': 'Gonzi cdc',
    'Tabak': 'Tabak cdc',
    'Romi':'Romi nor'
})
df_cena = df_cena[['id_cena','fecha','quorum','comida','precio','casa','tema']]
df_cena
df_cena.head()



,id_cena,fecha,quorum,comida,precio,casa,tema
0,1,2024-03-02,SI,Glorias,NaN,Glorias,NaN
1,2,2024-03-09,NO,Asado,NaN,Cabelli,NaN
2,3,2024-03-16,SI,Asado,9500.00,Saroka,NaN
3,4,2024-03-23,SI,Glorias,8500.00,Glorias,NaN
4,5,2024-03-30,SI,Asado,12000.00,Soko,NaN


In [8]:
# Sacás los nombres de las columnas 1 a 11 (los participantes)
nombres = df_mosca.columns[1:12]

# Creás el DataFrame con esos nombres, como una columna llamada 'participante'
df_persona = pd.DataFrame(nombres, columns=['nombre'])
df_persona


,nombre
0,axel zielonka
1,federico saroka
2,federico cabelli
3,federico koltan
4,manuel hirsch
5,joaquín sokolowicz
6,lucas rotmitrovsky
7,gonzalo borinsky
8,matias nemirovsky
9,ianick izon


In [9]:
personas_cols = df_mosca.columns[1:12]

# Convertir strings a float
df_mosca[personas_cols] = df_mosca[personas_cols].astype(float)

# Reemplazar 0.5 por 1 y 0.25 por 0
def convertir_valor(x):
    if x == 0.5:
        return 1
    elif x == 0.25:
        return 0
    else:
        return x

df_mosca[personas_cols] = df_mosca[personas_cols].applymap(convertir_valor).astype(int)




C:\Users\Saroka\AppData\Local\Temp\ipykernel_26700\3897130293.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_mosca[personas_cols] = df_mosca[personas_cols].applymap(convertir_valor).astype(int)


In [11]:
df_mosca["id_cena"] = range(1, len(df_mosca) + 1)
# Definir las columnas de personas (en tu ejemplo son las columnas desde la 2 hasta la 12)
personas_cols = df_mosca.columns[1:12]

# Crear una lista para almacenar los datos
filas_participacion = []

# Recorrer fila por fila
for _, fila in df_mosca.iterrows():
    id_cena = fila['id_cena']
    for persona in personas_cols:
        if fila[persona] == 1:
            filas_participacion.append({'id_cena': id_cena, 'nombre': persona})

# Crear el DataFrame final
df_cena_participante = pd.DataFrame(filas_participacion)
i=1
if i==1:
    for fila in df_cena_participante.id_cena: #ESTO DEPENDE DE CADA EXCEL
        df_cena_participante['distancia'] = np.nan
        df_cena_participante['ida'] = np.nan
        df_cena_participante['vuelta'] = np.nan
df_cena_participante.to_excel('cena_participante_2024.xlsx', index=False)

In [13]:
cenas_con_participantes = set(df_cena_participante['id_cena'].unique())

# Paso 2: Crear la columna 'realizada' en df_cena
df_cena['realizada'] = df_cena['id_cena'].apply(lambda x: 1 if x in cenas_con_participantes else 0)

# Paso 3: Contar participantes por cena
participantes_por_cena = df_cena_participante.groupby('id_cena').size()

# Paso 4: Asignar la cantidad a df_cena, para las cenas sin participantes poner 0
df_cena['cantidad_personas'] = df_cena['id_cena'].map(participantes_por_cena).fillna(0).astype(int)
df_cena = df_cena[['id_cena','fecha','realizada','quorum','comida','precio','casa','tema','cantidad_personas']]

df_cena['precio'] = df_cena['precio'].astype(float)
df_cena['casa'] = df_cena['casa'].replace('-',np.nan)

df_cena.head()
df_cena.to_excel('cena_2024.xlsx', index=False)

In [15]:
# Columnas de personas igual que antes
personas_cols = df_mosca.columns[1:12]

filas_inasistencia = []

for _, fila in df_mosca.iterrows():
    id_cena = fila['id_cena']
    for persona in personas_cols:
        if fila[persona] == 0:
            filas_inasistencia.append({'id_cena': id_cena, 'nombre': persona})

df_cena_inasistencia = pd.DataFrame(filas_inasistencia)
z=1
if z==1:# ESTO DEPENDE DE CADA EXCEL
    for fila in df_cena_inasistencia['nombre']:
        df_cena_inasistencia['razon'] = np.nan
df_cena_inasistencia.to_excel('cena_inasistencia_2024.xlsx', index=False)


In [29]:
df_cena = df_cena.applymap(
    lambda x: x.lower() if isinstance(x, str) else x
)


C:\Users\Saroka\AppData\Local\Temp\ipykernel_26700\1797364516.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_cena = df_cena.applymap(


In [32]:
casas = pd.read_csv('casa.csv')

In [35]:
casas_no_en_casas = (
    df_cena.loc[~df_cena["casa"].isin(casas["casa"]), "casa"]
    .dropna()
    .unique()
)

casas_no_en_casas


array(['glorias', 'romi nor', 'espacio tigre'], dtype=object)

In [36]:
import requests
import pandas as pd

# =========================
# CONFIG
# =========================
TZ = "America/Argentina/Buenos_Aires"

HOURLY_VARS = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "windspeed_10m",
]

SPANISH_COLS = {
    "temperature_2m": "temperatura_c",
    "apparent_temperature": "sensacion_termica_c",
    "relative_humidity_2m": "humedad_relativa_pct",
    "precipitation": "precipitacion_mm",
    "windspeed_10m": "velocidad_viento_kmh",
}

# =========================
# 1) Mapa de coords desde 'casas'
# =========================
# Normaliza nombres de casa en el mapa
coords_map = {
    str(r["casa"]).strip().lower(): (float(r["latitud"]), float(r["longitud"]))
    for _, r in casas[["casa", "latitud", "longitud"]].dropna().iterrows()
}

def get_coords_with_default(place_name, default="cdc"):
    name = str(place_name).strip().lower()
    if name in {"", "-", "nan", "none"}:
        return coords_map.get(default, None)
    return coords_map.get(name, coords_map.get(default, None))

# =========================
# 2) Fetch + promedio 21-23 del día + 00 del día siguiente
# =========================
weather_cache = {}  # (lat, lon, fecha, fecha_next) -> dict promedios

def fetch_weather_avg_21_to_00_nextday(lat, lon, fecha_str):
    fecha = pd.to_datetime(fecha_str).date()
    fecha_next = (pd.to_datetime(fecha_str) + pd.Timedelta(days=1)).date()

    key = (lat, lon, str(fecha), str(fecha_next))
    if key in weather_cache:
        return weather_cache[key]

    start_date = str(fecha)
    end_date = str(fecha_next)

    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly={','.join(HOURLY_VARS)}"
        f"&timezone=America%2FArgentina%2FBuenos_Aires"
    )

    resp = requests.get(url, timeout=30)
    data = resp.json()

    hourly = data.get("hourly", {})
    dfh = pd.DataFrame(hourly)
    dfh["time"] = pd.to_datetime(dfh["time"])

    mask_21_23 = (dfh["time"].dt.date == fecha) & (dfh["time"].dt.hour.isin([21, 22, 23]))
    mask_00_next = (dfh["time"].dt.date == fecha_next) & (dfh["time"].dt.hour == 0)

    sel = dfh[mask_21_23 | mask_00_next].copy()

    # fallback: si no aparece 00 del día siguiente, promedia solo 21-23 del mismo día
    if sel.empty or not mask_00_next.any():
        sel = dfh[mask_21_23].copy()

    out = {var: float(sel[var].mean()) for var in HOURLY_VARS}

    weather_cache[key] = out
    return out

# =========================
# 3) Aplicar a df_cena
# =========================
df_cena_clima = df_cena.copy()

df_cena_clima["fecha_str"] = pd.to_datetime(df_cena_clima["fecha"]).dt.strftime("%Y-%m-%d")

# crear columnas nuevas
for var, col_es in SPANISH_COLS.items():
    df_cena_clima[col_es] = pd.NA

for idx, row in df_cena_clima.iterrows():
    latlon = get_coords_with_default(row["casa"], default="cdc")
    if latlon is None:
        continue

    lat, lon = latlon
    fecha_str = row["fecha_str"]

    w = fetch_weather_avg_21_to_00_nextday(lat, lon, fecha_str)

    for var, col_es in SPANISH_COLS.items():
        val = w.get(var, None)
        if val is None:
            continue
        if var in {"temperature_2m", "apparent_temperature", "relative_humidity_2m", "windspeed_10m"}:
            df_cena_clima.at[idx, col_es] = round(val, 1)
        elif var == "precipitation":
            df_cena_clima.at[idx, col_es] = round(val, 2)

df_cena_clima = df_cena_clima.drop(columns=["fecha_str"])

# Si querés reemplazar el df original:
df_cena = df_cena_clima


In [39]:
df_cena.to_excel('cena_2024.xlsx', index=False)

In [19]:
df_casa = pd.DataFrame(df_cena['casa'].dropna().unique(), columns=['casa'])
w=1
if w==1:
    df_casa = df_casa.drop(index=[7,9]).reset_index(drop=True) # Esto depende de cada excel
    df_casa.loc[0,['latitud','longitud']] = ['-34.416625', '-58.612249']
    df_casa.loc[1,['latitud','longitud']] = ['-34.412331', '-58.671685']
    df_casa.loc[2,['latitud','longitud']] = ['-34.396986', '-58.618515']
    df_casa.loc[3,['latitud','longitud']] = ['-34.412336', '-58.613010']
    df_casa.loc[4,['latitud','longitud']] = ['-34.427644', '-58.608126']
    df_casa.loc[5,['latitud','longitud']] = ['-34.403045', '-58.649266']
    df_casa.loc[6,['latitud','longitud']] = ['-34.413042', '-58.614892']
    df_casa.loc[7,['latitud','longitud']] = ['-34.427644', '-58.608126']
    df_casa.loc[8,['latitud','longitud']] = ['-34.556047', '-58.458373']
    df_casa.loc[9,['latitud','longitud']] = ['-34.532358', '-58.474440']
    df_casa.loc[10,['casa','latitud','longitud']] = ['Ianick','-34.559861', '-58.445627']
    df_casa.loc[11,['casa','latitud','longitud']] = ['Axel capi','-34.564737', '-58.445774']
    df_casa.loc[12,['casa','latitud','longitud']] = ['Gonzi capi','-34.565775', '-58.442115']
    df_casa.loc[13,['casa','latitud','longitud']] = ['Tabak capi','-34.583843', '-58.421422']
    df_casa.loc[14,['casa','latitud','longitud']] = ['Romi capi','-34.557289', '-58.447149']
    df_casa.loc[15,['casa','latitud','longitud']] = ['Sheila','-34.403139', '-58.613479']
    df_casa.loc[16,['casa','latitud','longitud']] = ['Martu szkol','-34.548376', '-58.468273']
    

    
df_casa

,casa,latitud,longitud
0,Glorias,-34.416625,-58.612249
1,Cabelli,-34.412331,-58.671685
2,Saroka,-34.396986,-58.618515
3,Soko,-34.412336,-58.613010
4,Gonzi cdc,-34.427644,-58.608126
5,Romi nor,-34.403045,-58.649266
6,Tabak cdc,-34.413042,-58.614892
7,Espacio tigre,-34.427644,-58.608126
8,Manu,-34.556047,-58.458373
9,NaN,-34.532358,-58.474440


In [61]:
df_casa_posta = df_casa.loc[[1,2,3,8,10,11,12,13,14]]

df_casa_posta.reset_index(drop=True, inplace=True)
df_casa_posta.loc[len(df_casa_posta)] = ['Nemi',-34.556047,-58.458373]
df_casa_posta.loc[len(df_casa_posta)] = ['Koltan',-34.588124,-58.401844]

df_casa_posta
df_casa_posta.iloc[3,1]= '-34.532358'
df_casa_posta.iloc[3,2]= '-58.47444'
df_casa_posta['latitud'] = df_casa_posta['latitud'].astype(float)
df_casa_posta['longitud'] = df_casa_posta['longitud'].astype(float)

df_casa_posta

,casa,latitud,longitud
0,Cabelli,-34.412331,-58.671685
1,Saroka,-34.396986,-58.618515
2,Soko,-34.412336,-58.613010
3,Manu,-34.532358,-58.474440
4,Ianick,-34.559861,-58.445627
5,Axel capi,-34.564737,-58.445774
6,Gonzi capi,-34.565775,-58.442115
7,Tabak capi,-34.583843,-58.421422
8,Romi capi,-34.557289,-58.447149
9,Nemi,-34.556047,-58.458373


In [63]:
import folium

# Calcular centro
lat_media = df_casa_posta['latitud'].mean()
lon_media = df_casa_posta['longitud'].mean()

# Crear el mapa centrado en el punto medio
mapa = folium.Map(location=[lat_media, lon_media], zoom_start=12)

# Agregar los puntos de las casas
for _, row in df_casa_posta.iterrows():
    folium.Marker(
        location=[row["latitud"], row["longitud"]],
        popup=row["casa"],
        tooltip=row["casa"]
    ).add_to(mapa)

# Agregar el punto medio destacado
folium.Marker(
    location=[lat_media, lon_media],
    popup=f"Punto medio<br>Lat: {lat_media:.6f}<br>Lon: {lon_media:.6f}",
    tooltip="Punto medio",
    icon=folium.Icon(color='red', icon='star', prefix='fa')
).add_to(mapa)

# Mostrar mapa (en Jupyter) o guardar
mapa.save("mapa_interactivo.html")


In [233]:
df_cena = df_cena[['id_cena','fecha','realizada','quorum','comida','precio','casa','tema','cantidad_personas']]
df_casa
df_cena_inasistencia
for fila in df_cena_participante.id_cena:
    df_cena_participante['distancia'] = np.nan
    df_cena_participante['ida'] = np.nan
    df_cena_participante['vuelta'] = np.nan
df_cena_participante
df_persona = df_persona.rename(columns={'participante': 'nombre'})

In [228]:
df_cena

,id_cena,fecha,realizada,quorum,precio,casa,tema,cantidad_personas
0,1,2024-03-02,1,SI,NaN,Glorias,NaN,9
1,2,2024-03-09,1,NO,NaN,Cabelli,NaN,5
2,3,2024-03-16,1,SI,9500.0,Saroka,NaN,9
3,4,2024-03-23,1,SI,8500.0,Glorias,NaN,8
4,5,2024-03-30,1,SI,12000.0,Soko,NaN,6
5,6,2024-04-06,1,SI,8000.0,Gonzi cdc,NaN,8
6,7,2024-04-13,1,SI,9500.0,Romi nor,Make it memes,9
7,8,2024-04-20,1,SI,8500.0,Saroka,Make it memes,7
8,9,2024-04-27,1,SI,7800.0,Tabak cdc,NaN,9
9,10,2024-05-04,0,NO,NaN,NaN,-,0


ACA SUBO LOS DATOS A SQL

In [ ]:
from sqlalchemy import create_engine

# Parámetros de conexión
server = 'LAPTOP-7C6OKS2O\\SQLEXPRESS'
database = 'Cenas_mosca'

# Cadena de conexión para Windows Authentication
connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Crear el engine
engine = create_engine(connection_string)

# Subir los DataFrames (con columnas ya renombradas y dtypes correctos)
#df_persona.to_sql('participante', engine, if_exists='append', index=False)
#df_casa.to_sql('casa', engine, if_exists='append', index=False)
#df_cena.to_sql('cena', engine, if_exists='append', index=False)
#df_cena_participante.to_sql('cena_participante', engine, if_exists='append', index=False)
#df_cena_inasistencia.to_sql('cena_inasistencia', engine, if_exists='append', index=False)


189

In [238]:
import pandas as pd
from sqlalchemy import create_engine
import pyodbc

# Conexión
server = 'LAPTOP-7C6OKS2O\\SQLEXPRESS'
database = 'Cenas_mosca'
connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
engine = create_engine(connection_string)

# Suponiendo que df_cena ya tiene las columnas: id_cena y comida
# Iteramos y actualizamos uno por uno
with engine.begin() as connection:
    for index, row in df_cena.iterrows():
        id_cena = int(row['id_cena'])
        comida = row['comida'] if pd.notnull(row['comida']) else None
        connection.execute(
            "UPDATE cena SET comida = :comida WHERE id_cena = :id_cena",
            {"comida": comida, "id_cena": id_cena}
        )



ObjectNotExecutableError: Not an executable object: 'UPDATE cena SET comida = :comida WHERE id_cena = :id_cena'

In [235]:
df_cena['comida'] = df_cena['comida'].replace('-', np.nan)

In [223]:
df_cena

,id_cena,fecha,quorum,comida,precio,casa,tema
0,1,2024-03-02,SI,Glorias,NaN,Glorias,NaN
1,2,2024-03-09,NO,Asado,NaN,Cabelli,NaN
2,3,2024-03-16,SI,Asado,9500.00,Saroka,NaN
3,4,2024-03-23,SI,Glorias,8500.00,Glorias,NaN
4,5,2024-03-30,SI,Asado,12000.00,Soko,NaN
5,6,2024-04-06,SI,Pizzas y empanadas,8000.00,Gonzi cdc,NaN
6,7,2024-04-13,SI,Asado,9500.00,Romi nor,Make it memes
7,8,2024-04-20,SI,Cosco,8500.00,Saroka,Make it memes
8,9,2024-04-27,SI,Sinchi,7800.00,Tabak cdc,NaN
9,10,2024-05-04,NO,-,NaN,-,-
